In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Text Cleaning


In [ ]:
import re
train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

# Embeddings & Similarity 

In [9]:
import nltk
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
train["prompt_token"] = train["cleaned_prompt"].apply(nltk.word_tokenize)
for opt in ["A","B","C","D","E"]:
    train[f"{opt}_token"] = train[f"cleaned_{opt}"].apply(nltk.word_tokenize)

text = train["cleaned_prompt"].tolist()
for opt in ["A","B","C","D","E"]:
    text += train[f"cleaned_{opt}"].tolist()

tfidf = TfidfVectorizer()
tfidf.fit(text)

i = 0
p_vec = tfidf.transform([train.loc[i,"cleaned_prompt"]])
o_vec = [tfidf.transform([train.loc[i,f"cleaned_{opt}"]]) for opt in ["A","B","C","D","E"]]

cos_sim = [cosine_similarity(p_vec, opt)[0][0] for opt in o_vec]
print(cos_sim)

all_tokens = train["prompt_token"].tolist()
for opt in ["A","B","C","D","E"]:
    all_tokens += train[f"{opt}_token"].tolist()

w2v = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)
def avg_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) > 0:
        return np.mean(vecs, axis=0) 
    else:
        return np.zeros(model.vector_size)
        
i = 0
p_avg = avg_vector(train.loc[i,"prompt_token"], w2v)
o_avg = [avg_vector(train.loc[i,f"{opt}_token"], w2v) for opt in ["A","B","C","D","E"]]

w2v_sim = [cosine_similarity([p_avg],[o])[0][0] for o in o_avg]
print(w2v_sim)

def predict_top3(prompt_tokens, option_tokens, model):
    p_vec = avg_vector(prompt_tokens, model)
    sim = [cosine_similarity([p_vec],[avg_vector(opt, model)])[0][0] for opt in option_tokens]
    ranked = np.argsort(sim)[::-1]
    return [chr(65 + idx) for idx in ranked[:3]]
    
def mapk(actual, predicted, k=3):
    score = 0.0
    n = len(actual)

    for i in range(n):
        a = actual[i]
        p = predicted[i]
        if a in p:
            idx = p.index(a)
            score += 1.0 / (idx + 1)
    return score / n
    
actual_ans = train["answer"].tolist()[:500]
pred_ans = []

for i in range(500):
    prompt_tokens = train.loc[i,"prompt_token"]
    option_tokens_list = [train.loc[i,f"{opt}_token"] for opt in ["A","B","C","D","E"]]
    preds = predict_top3(prompt_tokens, option_tokens_list, w2v)
    pred_ans.append(preds)

print(mapk(actual_ans, pred_ans))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[np.float64(0.16679528994578402), np.float64(0.18754309767044808), np.float64(0.4200856621712537), np.float64(0.37036798616566935), np.float64(0.10824678045170937)]
[np.float32(0.7076273), np.float32(0.72048795), np.float32(0.8290952), np.float32(0.7431863), np.float32(0.6633757)]
0.34100000000000036


In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

rows = []
for i in range(len(train)):
    prompt = train.loc[i,"cleaned_prompt"]
    for opt in ["A","B","C","D","E"]:
        text_pair = prompt + " " + train.loc[i,f"cleaned_{opt}"]
        label = 1 if train.loc[i,"answer"] == opt else 0
        rows.append({"text": text_pair, "label": label})

dataset = Dataset.from_list(rows)
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2)
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

In [ ]:
sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample.to_csv('submission.csv', index=False)